# TRELLIS — Imagen → 3D (GLB)

**Requisito:** Runtime → Cambiar tipo de entorno de ejecución → **T4 GPU**

Tiempo estimado:
- Instalación: ~8-12 min (solo la primera vez)
- Generación: ~2-4 min por imagen

In [ ]:
# ── Celda 1: Verificar GPU ──────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NO GPU — cambia el runtime a T4!')

In [ ]:
# ── Celda 2: Instalar dependencias base ─────────────────────────────────
# Tarda ~5 min — ejecutar solo una vez por sesión
import os

# Wheel de flash-attn precompilado para CUDA 12.x + Python 3.10 (Colab default)
os.system('pip install -q packaging ninja')
os.system('pip install -q flash-attn --no-build-isolation')

# spconv (sparse convolutions para TRELLIS)
os.system('pip install -q spconv-cu120')

# Resto de deps de TRELLIS
os.system('pip install -q \
    diffusers transformers accelerate \
    trimesh[easy] \
    imageio[ffmpeg] \
    xatlas \
    pymeshlab \
    open3d \
    huggingface_hub \
    pillow numpy scipy einops')

# nvdiffrast (diferenciable rasterizer)
os.system('pip install -q git+https://github.com/NVlabs/nvdiffrast.git')

print('Dependencias instaladas.')

In [ ]:
# ── Celda 3: Clonar e instalar TRELLIS ──────────────────────────────────
import os

if not os.path.exists('/content/TRELLIS'):
    os.system('git clone --recurse-submodules https://github.com/microsoft/TRELLIS.git /content/TRELLIS')

os.chdir('/content/TRELLIS')
os.system('pip install -q -e . --no-deps')
print('TRELLIS listo.')

In [ ]:
# ── Celda 4: Cargar pipeline ─────────────────────────────────────────────
# Descarga pesos ~6 GB la primera vez (se cachean en /root/.cache/huggingface)
import sys
sys.path.insert(0, '/content/TRELLIS')

from trellis.pipelines import TrellisImageTo3DPipeline

print('Cargando modelo (puede tardar ~2 min en descarga + carga)...')
pipeline = TrellisImageTo3DPipeline.from_pretrained('JeffreyXiang/TRELLIS-image-large')
pipeline = pipeline.cuda()
print('Pipeline listo.')

In [ ]:
# ── Celda 5: Subir imagen y generar 3D ───────────────────────────────────
from google.colab import files
from PIL import Image
import io

print('Selecciona tu imagen:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
image = Image.open(io.BytesIO(uploaded[filename])).convert('RGBA')
print(f'Imagen cargada: {filename}  {image.size}')

In [ ]:
# ── Celda 6: Generar modelo ──────────────────────────────────────────────
# Parámetros ajustables
SEED = 1
SS_GUIDANCE     = 7.5
SS_STEPS        = 12
SLAT_GUIDANCE   = 3.0
SLAT_STEPS      = 12
MESH_SIMPLIFY   = 0.95
TEXTURE_SIZE    = 1024

import torch
from pathlib import Path

print('Generando... (~2-4 min)')
with torch.no_grad():
    outputs = pipeline.run(
        image,
        seed=SEED,
        formats=['mesh'],
        preprocess_image=True,
        sparse_structure_sampler_params={
            'steps': SS_STEPS,
            'cfg_strength': SS_GUIDANCE,
        },
        slat_sampler_params={
            'steps': SLAT_STEPS,
            'cfg_strength': SLAT_GUIDANCE,
        },
    )

print('Extrayendo GLB...')
stem = Path(filename).stem
glb_path = f'/content/{stem}.glb'

glb = outputs['mesh'][0].export_glb(
    simplify=MESH_SIMPLIFY,
    texture_size=TEXTURE_SIZE,
)
with open(glb_path, 'wb') as f:
    f.write(glb)

size_kb = Path(glb_path).stat().st_size // 1024
print(f'Listo. {glb_path}  ({size_kb} KB)')

In [ ]:
# ── Celda 7: Descargar GLB ───────────────────────────────────────────────
from google.colab import files
files.download(glb_path)
print('Descargando', glb_path)

In [ ]:
# ── Celda 8 (opcional): Preview 3D en Colab ──────────────────────────────
# Requiere: pip install trimesh[easy] pythreejs (ya instalado arriba)
import trimesh
from IPython.display import display

mesh = trimesh.load(glb_path)
scene = mesh.scene() if hasattr(mesh, 'scene') else trimesh.Scene(mesh)
display(scene.show(viewer='notebook'))